# Stage 3 + Tasks 3B/3C — Multi-file revised pipeline

Run after revised Stage 1 and Stage 2 have completed. Heddaya outcomes are assumed to be `sale`; AUC is skipped for Heddaya because it has only one outcome class.

In [ ]:
# Optional: mount Drive if needed
# from google.colab import drive
# drive.mount("/content/drive")

import os
os.makedirs("/content/optimal-nego/data", exist_ok=True)
os.makedirs("/content/optimal-nego/outputs", exist_ok=True)
print("Expected raw files in /content/optimal-nego/data:")
print("- dealerships-nego.csv")
print("- emaad-sales.csv")
print("- heddaya-nego.csv")

## Stage 3: Bargaining act annotation

In [ ]:
"""
Stage 3: Bargaining Act Annotation + Layer 2 Sub-typing
Revised for multi-file Stage 1/2 pipeline

Inputs supported:
  /content/optimal-nego/data/dealerships-nego.csv
  /content/optimal-nego/data/emaad-sales.csv
  /content/optimal-nego/data/heddaya-nego.csv

Expected Stage 1/2 layout:
  /content/optimal-nego/outputs/<dataset_key>/...

Output per dataset:
  /content/optimal-nego/outputs/<dataset_key>/stage3_annotated.csv
  /content/optimal-nego/outputs/<dataset_key>/stage3_diagnostics.txt

Notes:
  - Heddaya schema is standardized from conv_id/word/role/duration_min.
  - Heddaya outcome is assumed to be sale for every conversation.
  - Role is taken from an existing role column where available; otherwise inferred.
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR = "/content/optimal-nego"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATASETS = [
    {
        "key": "dealerships_nego",
        "path": f"{DATA_DIR}/dealerships-nego.csv",
        "assume_outcome": None,
    },
    {
        "key": "emaad_sales",
        "path": f"{DATA_DIR}/emaad-sales.csv",
        "assume_outcome": None,
    },
    {
        "key": "heddaya_nego",
        "path": f"{DATA_DIR}/heddaya-nego.csv",
        "assume_outcome": "sale",
    },
]

BC_RE = re.compile(
    r"^\s*(yeah|mhm|mm|um|uh|okay|ok|yes|no|hi|hello|so|right|"
    r"sure|huh|ah|oh|hmm|yep|nope|alright|well|hey)\s*$",
    re.IGNORECASE,
)
PRICE_HINT_RE = re.compile(r"\$?\d[\d,\.]+|\bprice\b|\boffer\b|\bbid\b|\bout[- ]?the[- ]?door\b|\botd\b", re.IGNORECASE)

# Known non-price numbers in the Heddaya-style housing corpus plus generic distractors.
NON_PRICE = {
    "1846", "1715", "1875", "1920", "90", "89", "13878", "06898", "04725", "08614", "1947",
    "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12",
}

# -----------------------------
# Schema standardization
# -----------------------------

def normalize_outcome(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace("_", " ").replace("-", " ")
    if s in {"sale", "deal", "sold", "yes", "1", "true", "closed", "success"}:
        return "sale"
    if s in {"no sale", "nosale", "no deal", "nodeal", "no", "0", "false", "failed", "failure"}:
        return "no sale"
    return s


def normalize_role(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in {"buyer", "b", "customer", "client", "consumer", "0"}:
        return "buyer"
    if s in {"seller", "dealer", "salesperson", "sales", "agent", "d", "1"}:
        return "seller"
    return None


def standardize_raw(df, dataset_key, assume_outcome=None):
    df = df.copy()

    # Conversation id
    if "conversation_id" not in df.columns:
        if "conv_id" in df.columns:
            df = df.rename(columns={"conv_id": "conversation_id"})
        else:
            raise ValueError(f"{dataset_key}: no conversation_id or conv_id column found")

    # Text
    if "text" not in df.columns:
        if "word" in df.columns:
            df = df.rename(columns={"word": "text"})
        else:
            raise ValueError(f"{dataset_key}: no text or word column found")

    # Turn ordering / times
    if "start_time" not in df.columns:
        if "turn_id" in df.columns:
            df["start_time"] = df["turn_id"].astype(float)
        else:
            df["start_time"] = df.groupby("conversation_id").cumcount().astype(float)

    if "end_time" not in df.columns:
        if "duration_min" in df.columns:
            # Convert duration_min to an approximate cumulative minute timeline per conversation.
            # This preserves ordering and gives Stage 3 a stable sortable column.
            dur = pd.to_numeric(df["duration_min"], errors="coerce").fillna(0.0)
            df["end_time"] = df["start_time"].astype(float) + dur
        else:
            df["end_time"] = df["start_time"].astype(float) + 1.0

    # Speaker id
    if "speaker_id" not in df.columns:
        if "role" in df.columns:
            df["speaker_id"] = df["role"].map(lambda r: 0 if normalize_role(r) == "buyer" else 1)
        else:
            df["speaker_id"] = 0

    # Outcome
    if assume_outcome is not None:
        df["outcome"] = assume_outcome
    elif "outcome" in df.columns:
        df["outcome"] = df["outcome"].map(normalize_outcome)
    else:
        df["outcome"] = np.nan

    df["text"] = df["text"].fillna("").astype(str)
    df["conversation_id"] = df["conversation_id"].astype(str)
    df["speaker_id"] = pd.to_numeric(df["speaker_id"], errors="coerce").fillna(-1).astype(int)
    df["start_time"] = pd.to_numeric(df["start_time"], errors="coerce").fillna(0.0)
    df["end_time"] = pd.to_numeric(df["end_time"], errors="coerce").fillna(df["start_time"] + 1.0)

    # Preserve role if present; infer below when absent.
    if "role" in df.columns:
        df["role"] = df["role"].map(normalize_role)
    else:
        df["role"] = None

    df = df.sort_values(["conversation_id", "start_time", "end_time"]).reset_index(drop=True)
    df["is_bc"] = df["text"].apply(lambda t: len(t.strip()) <= 3 or bool(BC_RE.match(t)))
    return df


def infer_roles(df):
    """Use existing role if present; otherwise infer buyer/seller per conversation."""
    df = df.copy()
    if df["role"].notna().mean() > 0.5:
        return df

    role_map = {}
    for cid, grp in df.groupby("conversation_id"):
        buyer = None
        for _, row in grp.sort_values("start_time").iterrows():
            if PRICE_HINT_RE.search(row["text"]):
                buyer = row["speaker_id"]
                break
        if buyer is None:
            buyer = sorted(grp["speaker_id"].dropna().unique().tolist())[0] if len(grp) else 0
        spks = grp["speaker_id"].dropna().unique().tolist()
        sellers = [s for s in spks if s != buyer]
        seller = sellers[0] if sellers else (1 - int(buyer) if int(buyer) in [0, 1] else 1)
        role_map[cid] = {buyer: "buyer", seller: "seller"}

    df["role"] = df.apply(lambda r: role_map.get(r["conversation_id"], {}).get(r["speaker_id"], "buyer"), axis=1)
    return df

# -----------------------------
# Price extraction and acts
# -----------------------------

def extract_prices(text):
    prices = set()
    text = str(text)

    # Explicit dollar amounts: $23,500, $228,000, $228
    for m in re.finditer(r"\$\s*(\d[\d,\s\.]*)(?:\s*(k|K|thousand))?", text):
        raw = re.sub(r"[\s,]", "", m.group(1))
        try:
            val = float(raw)
            if m.group(2) or val < 1000:
                # Preserve 150-300 as thousands; otherwise keep normalized thousands.
                v = int(round(val))
            else:
                v = int(round(val / 1000))
            if 5 <= v <= 300:
                prices.add(v)
        except Exception:
            pass

    # Bare 3-digit values for housing-style prices.
    for m in re.finditer(r"\b(\d{3})\b", text):
        raw = m.group(1)
        if raw in NON_PRICE:
            continue
        try:
            v = int(raw)
            if 150 <= v <= 300:
                prices.add(v)
        except Exception:
            pass

    # 5-6 digit amounts: 23500, 228000
    for m in re.finditer(r"\b(\d{5,6})\b", text):
        try:
            value = int(m.group(1))
            # Normalize to thousands. 23500 -> 24, 228000 -> 228.
            v = int(round(value / 1000))
            if 5 <= v <= 300:
                prices.add(v)
        except Exception:
            pass

    # Spaced housing format: 2 28 -> 228
    for m in re.finditer(r"\b(2)\s+(\d{2})\b", text):
        try:
            v = int(m.group(1) + m.group(2))
            if 150 <= v <= 300:
                prices.add(v)
        except Exception:
            pass

    return prices

PUSH_PATTERNS = [
    r"\bcome down\b", r"\bgo up\b", r"\bgo higher\b", r"\bgo lower\b",
    r"\bmeet me\b", r"\bmeet halfway\b", r"\bcloser to\b", r"\ba little (bit )?(more|higher|lower|closer)\b",
    r"\bnot going to work\b", r"\bdoesn'?t work\b", r"\bwon'?t work\b", r"\bnot acceptable\b", r"\bcannot accept\b", r"\bcan'?t accept\b",
    r"\bstand firm\b", r"\bstick(ing)? (with|to)\b", r"\bhold(ing)? firm\b", r"\bfinal offer\b", r"\bbottom line\b",
    r"\bhighest (I'?m|i'?m|we'?re) willing\b", r"\blowest (I'?m|i'?m|we'?re) willing\b",
    r"\bcan you (do|go|come|meet|try|work)\b", r"\bwould you (consider|be willing|go|come|do)\b", r"\bif you (could|can|would)\b",
    r"\bgive (me|us) a (little|bit|few)\b", r"\bneed (you to|to) (move|come|go|do|work)\b",
    r"\bunreasonable\b", r"\breasonable\b", r"\bfair(ly)?\b", r"\bnot fair\b", r"\bwalk away\b", r"\bno deal\b",
    r"\bbest I can do\b", r"\bas (low|high) as (I|we)'?(ll|d) go\b",
]
PUSH_RE = re.compile("|".join(PUSH_PATTERNS), re.IGNORECASE)

COMP_PATTERNS = [
    r"\blisting\b", r"\bappendix\b", r"\b(89|90)\s*[\-–]?\s*\d+\b",
    r"\b(comparable|similar|neighboring|nearby|other)\s+(home|house|propert|listing|vehicle|car|dealer|deal)\b",
    r"\b(homes?|houses?|properties|cars?|vehicles?)\s+(in the area|nearby|around here|in the neighborhood|online|elsewhere)\b",
    r"\bother\s+(homes?|houses?|properties|cars?|vehicles?|dealers?)\b",
    r"\bmarket\s*(price|value|rate|data|research|analysis)?\b", r"\bselling (for|at|price)\b", r"\bsold (for|at|recently)\b",
    r"\brecently sold\b", r"\bsale (price|value)\b", r"\bper square (foot|feet|ft)\b", r"\bsquare (foot|feet|ft)\b", r"\bprice per\b",
    r"\bcompared? (to|with)\b", r"\bin comparison\b", r"\bsimilar(ly)?\b", r"\bjust like\b",
    r"\b213\b", r"\b(233|233,?000)\b", r"\b(239|239,?000)\b", r"\b1[,\s]?715\b", r"\b1[,\s]?875\b", r"\b1[,\s]?920\b",
    r"\b(in the|this|the) (neighborhood|area|region|vicinity|market)\b", r"\bhomes? around\b", r"\bproperties around\b",
]
COMP_RE = re.compile("|".join(COMP_PATTERNS), re.IGNORECASE)

ALLOW_HEDGE_RE = re.compile(
    r"\b(willing to (go|come|do|say|try|offer|accept|consider))\b"
    r"|\b(I'?ll (do|go|try|offer|say|come down to|come up to))\b"
    r"|\b(we could (do|go|say|offer|try))\b"
    r"|\b(let'?s (say|do|try|go with|settle on))\b"
    r"|\b(how about|what about|would .+ work)\b"
    r"|\b(meet in the middle|split the difference)\b"
    r"|\b(could (do|go|say|come down|come up|offer))\b"
    r"|\b(drop(ping)? (it |it'?s |down )?to)\b"
    r"|\b(raise|bump|move) (it |up )?(to)\b",
    re.IGNORECASE,
)

END_RE = re.compile(
    r"\b(deal|agreed?|done|sold|you'?ve got (yourself )?a deal|we have a deal|sounds good|works (for me|for us)|"
    r"I'?ll take (it|that)|I'?ll do (it|that)|that works|perfect|alright (then|[,.])|let'?s do (it|that)|shake on it|you got (it|yourself a deal))\b",
    re.IGNORECASE,
)

CONSTRAINT_RE = re.compile(
    r"\b(can'?t|cannot|won'?t|will not)\s+(go|pay|do|offer|afford|accept)\b|\b(budget|afford(able)?|maximum|limit(ed)?|ceiling|cap)\b|"
    r"\b(not (possible|doable|feasible) for (me|us))\b|\b(beyond|outside|above)\s+(my|our)\s+(budget|means|limit|range)\b|"
    r"\b(hard (stop|limit|no))\b|\b(stretch(ing)? (my|our|the))\b|\b(as (high|low|much|far) as (I|we)('?(ll|d))? (go|do|offer))\b|"
    r"\b(that'?s (the most|all) (I|we) can)\b|\bwalk away\b",
    re.IGNORECASE,
)
DISPARAGE_RE = re.compile(
    r"\b(overpriced|over-priced|over priced)\b|\b(not worth|isn'?t worth|doesn'?t justify)\b|\b(too (high|much|expensive))\b|"
    r"\b(excessive(ly)?)\b|\b(inflated)\b|\b(not (justified|warranted))\b|\b(asking too much)\b|\b(price(d)? too)\b|"
    r"\b(doesn'?t (reflect|match) the (value|market))\b",
    re.IGNORECASE,
)
COMP_PRICE_RE = re.compile(
    r"\b(selling for|sold for|sale price|asking price|listed (at|for)|went for|priced at|market (price|value|rate)|price per|cost per)\b|"
    r"\b(per square (foot|feet|ft))\b|\b213\b|\b(233|239)\b",
    re.IGNORECASE,
)
COMP_QUALITY_RE = re.compile(
    r"\b(square (foot|feet|ft)|sq\.?\s?ft\.?|sqft)\b|\b(bedroom|bathroom|garage|kitchen|fireplace|hardwood|landscap|renovated|updated|decorated|appliance)\b|"
    r"\b(size|space|room|floor|yard|backyard|condition|feature|features|trim|mileage|color|warranty|package|amenity|amenities)\b|"
    r"\b(1[,\s]?715|1[,\s]?875|1[,\s]?920|1[,\s]?846)\b",
    re.IGNORECASE,
)


def annotate_conversation(turns):
    results = []
    all_prices_seen = set()
    spk_last_offer = {}

    for _, row in turns.iterrows():
        text = str(row["text"])
        spk = int(row["speaker_id"])
        turn_prices = extract_prices(text)
        acts = set()

        if END_RE.search(text) and len(text.strip()) > 5:
            acts.add("end")

        for p in turn_prices:
            if p not in all_prices_seen:
                acts.add("new_offer")
            else:
                acts.add("repeat_offer")

        if turn_prices and ALLOW_HEDGE_RE.search(text):
            other_candidates = [s for s in spk_last_offer if s != spk]
            other_spk = other_candidates[-1] if other_candidates else None
            if other_spk is not None:
                other_price = spk_last_offer.get(other_spk)
                my_price = spk_last_offer.get(spk)
                for p in turn_prices:
                    if my_price is not None and other_price is not None:
                        if ((p > my_price and p <= other_price) or (p < my_price and p >= other_price)):
                            acts.discard("new_offer")
                            acts.add("allowance")
                            break
            else:
                acts.discard("new_offer")
                acts.add("allowance")

        if PUSH_RE.search(text):
            acts.add("push")
        if COMP_RE.search(text):
            acts.add("comparison")

        if turn_prices:
            all_prices_seen.update(turn_prices)
            spk_last_offer[spk] = sorted(turn_prices)[0]

        push_subtype = None
        comp_subtype = None
        if "push" in acts:
            is_constraint = bool(CONSTRAINT_RE.search(text))
            is_disparage = bool(DISPARAGE_RE.search(text))
            if is_constraint:
                push_subtype = "push_constraint"
            elif is_disparage:
                push_subtype = "push_disparagement"
            else:
                push_subtype = "push_neutral"

        if "comparison" in acts:
            is_price = bool(COMP_PRICE_RE.search(text))
            is_quality = bool(COMP_QUALITY_RE.search(text))
            if is_price and is_quality:
                comp_subtype = "comparison_mixed"
            elif is_price:
                comp_subtype = "comparison_price"
            elif is_quality:
                comp_subtype = "comparison_quality"
            else:
                comp_subtype = "comparison_price"

        results.append({
            "conversation_id": row["conversation_id"],
            "speaker_id": spk,
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "text": text,
            "outcome": row.get("outcome", np.nan),
            "role": row.get("role", None),
            "is_bc": bool(row.get("is_bc", False)),
            "act_new_offer": int("new_offer" in acts),
            "act_repeat": int("repeat_offer" in acts),
            "act_push": int("push" in acts),
            "act_comparison": int("comparison" in acts),
            "act_allowance": int("allowance" in acts),
            "act_end": int("end" in acts),
            "push_subtype": push_subtype,
            "comp_subtype": comp_subtype,
            "n_acts": len(acts),
            "acts_list": "|".join(sorted(acts)) if acts else "none",
        })

    return results


def annotate_dataset(spec):
    key = spec["key"]
    out_dir = f"{OUTPUT_ROOT}/{key}"
    os.makedirs(out_dir, exist_ok=True)
    diag = []

    def log(s=""):
        print(s)
        diag.append(str(s))

    log("=" * 72)
    log(f"STAGE 3: BARGAINING ACT ANNOTATION — {key}")
    log("=" * 72)

    if not os.path.exists(spec["path"]):
        log(f"SKIPPED: data file not found: {spec['path']}")
        with open(f"{out_dir}/stage3_diagnostics.txt", "w") as f:
            f.write("\n".join(diag))
        return None

    raw = pd.read_csv(spec["path"])
    df = standardize_raw(raw, key, assume_outcome=spec.get("assume_outcome"))
    df = infer_roles(df)

    log(f"[Step 1] Loaded and standardized data")
    log(f"  Total turns       : {len(df):,}")
    log(f"  Conversations     : {df['conversation_id'].nunique():,}")
    log(f"  Substantive turns : {(~df['is_bc']).sum():,}")
    log(f"  Backchannels      : {df['is_bc'].sum():,}")
    if spec.get("assume_outcome") is not None:
        log(f"  Outcome assumption: all conversations set to {spec['assume_outcome']}")
    else:
        log(f"  Outcome values    : {sorted([str(x) for x in df['outcome'].dropna().unique()])}")

    log("\n[Step 2] Annotating bargaining acts...")
    all_results = []
    for _, grp in df.groupby("conversation_id", sort=False):
        all_results.extend(annotate_conversation(grp.sort_values(["start_time", "end_time"])))
    ann = pd.DataFrame(all_results)

    sub = ann[~ann["is_bc"]].copy()
    total_sub = max(len(sub), 1)
    acts_cols = ["act_new_offer", "act_repeat", "act_push", "act_comparison", "act_allowance", "act_end"]
    act_names = ["New Offer", "Repeat Offer", "Push", "Comparison", "Allowance", "End"]

    log("\n[Step 3] Layer 1 coverage:")
    for col, name in zip(acts_cols, act_names):
        n = int(sub[col].sum())
        log(f"  {name:<15}: {n:5d} turns ({n / total_sub * 100:5.1f}%)")

    multi = int((sub[acts_cols].sum(axis=1) > 1).sum())
    noact = int((sub[acts_cols].sum(axis=1) == 0).sum())
    log(f"\n  Multi-act turns   : {multi:5d} ({multi / total_sub * 100:5.1f}%)")
    log(f"  No-act turns      : {noact:5d} ({noact / total_sub * 100:5.1f}%)")

    log("\n[Step 4] Layer 2 subtype distributions:")
    push_turns = sub[sub["act_push"] == 1]
    for sub_t in ["push_constraint", "push_disparagement", "push_neutral"]:
        n = int((push_turns["push_subtype"] == sub_t).sum())
        pct = n / len(push_turns) * 100 if len(push_turns) else 0
        log(f"  {sub_t:<25}: {n:5d} ({pct:5.1f}% of Push)")
    comp_turns = sub[sub["act_comparison"] == 1]
    for sub_t in ["comparison_price", "comparison_quality", "comparison_mixed"]:
        n = int((comp_turns["comp_subtype"] == sub_t).sum())
        pct = n / len(comp_turns) * 100 if len(comp_turns) else 0
        log(f"  {sub_t:<25}: {n:5d} ({pct:5.1f}% of Comparison)")

    log("\n[Step 5] Buyer vs Seller profiles:")
    for role in ["buyer", "seller"]:
        role_turns = sub[sub["role"] == role]
        denom = max(len(role_turns), 1)
        log(f"\n  {role.upper()} ({len(role_turns):,} turns):")
        for col, name in zip(acts_cols, act_names):
            n = int(role_turns[col].sum())
            log(f"    {name:<15}: {n:5d} ({n / denom * 100:5.1f}%)")

    if sub["outcome"].notna().any():
        log("\n[Step 6] Act frequency by outcome:")
        for col, name in zip(["act_push", "act_comparison"], ["Push", "Comparison"]):
            sale = sub[sub["outcome"] == "sale"][col].mean() * 100 if (sub["outcome"] == "sale").any() else np.nan
            nosale = sub[sub["outcome"] == "no sale"][col].mean() * 100 if (sub["outcome"] == "no sale").any() else np.nan
            log(f"  {name}: sale={sale if pd.notna(sale) else float('nan'):.1f}%  no-sale={nosale if pd.notna(nosale) else float('nan'):.1f}%")

    out_csv = f"{out_dir}/stage3_annotated.csv"
    ann.to_csv(out_csv, index=False)
    log(f"\n[Step 7] stage3_annotated.csv saved: {ann.shape}")
    log(f"  Path: {out_csv}")

    with open(f"{out_dir}/stage3_diagnostics.txt", "w") as f:
        f.write("\n".join(diag))

    log("=" * 72)
    log(f"STAGE 3 COMPLETE — {key}")
    log("=" * 72)
    return ann


def main():
    outputs = {}
    for spec in DATASETS:
        ann = annotate_dataset(spec)
        outputs[spec["key"]] = ann
        print()
    return outputs

if __name__ == "__main__":
    outputs = main()


## Tasks 3B + 3C: Tactic × latent state overlay and horse-race regression

In [ ]:
"""
Stage 3B + 3C: Tactic × Latent State Overlay and Horse Race Regression
Revised for multi-file Stage 1/2/3 pipeline

Reads per-dataset files:
  /content/optimal-nego/outputs/<dataset_key>/stage3_annotated.csv
  /content/optimal-nego/outputs/<dataset_key>/stage2_latent_states.csv

Writes per-dataset files:
  task3b_tactic_shifts.csv
  task3c_horse_race.csv              [if both outcome classes exist]
  task3c_features.csv
  task3bc_diagnostics.txt

Heddaya assumption:
  All outcomes are sale. The script still generates tactic-shift and feature files,
  but skips ROC-AUC horse-race evaluation because AUC requires both classes.
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR = "/content/optimal-nego"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"

DATASETS = ["dealerships_nego", "emaad_sales", "heddaya_nego"]

# -----------------------------
# Loading and merging
# -----------------------------

def find_z_cols(latent):
    z_cols = [c for c in latent.columns if re.fullmatch(r"z_\d+", str(c))]
    return sorted(z_cols, key=lambda x: int(x.split("_")[1]))


def normalize_outcome(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace("_", " ").replace("-", " ")
    if s in {"sale", "deal", "sold", "yes", "1", "true", "closed", "success"}:
        return "sale"
    if s in {"no sale", "nosale", "no deal", "nodeal", "no", "0", "false", "failed", "failure"}:
        return "no sale"
    return s


def load_merged(dataset_key):
    out_dir = f"{OUTPUT_ROOT}/{dataset_key}"
    ann_path = f"{out_dir}/stage3_annotated.csv"
    latent_path = f"{out_dir}/stage2_latent_states.csv"

    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"Missing Stage 3 annotations: {ann_path}")
    if not os.path.exists(latent_path):
        raise FileNotFoundError(f"Missing Stage 2 latent states: {latent_path}")

    ann = pd.read_csv(ann_path)
    latent = pd.read_csv(latent_path)

    ann["conversation_id"] = ann["conversation_id"].astype(str)
    latent["conversation_id"] = latent["conversation_id"].astype(str)

    if "is_bc" in ann.columns:
        ann_sub = ann[~ann["is_bc"].astype(bool)].copy()
    else:
        ann_sub = ann.copy()

    sort_cols = ["conversation_id"]
    if "start_time" in ann_sub.columns:
        sort_cols.append("start_time")
    if "end_time" in ann_sub.columns:
        sort_cols.append("end_time")
    ann_sub = ann_sub.sort_values(sort_cols).reset_index(drop=True)
    ann_sub["turn"] = ann_sub.groupby("conversation_id").cumcount()

    if "turn" not in latent.columns:
        latent = latent.sort_values(["conversation_id"]).reset_index(drop=True)
        latent["turn"] = latent.groupby("conversation_id").cumcount()

    # Avoid duplicate/ambiguous outcome columns.
    latent_cols = [c for c in latent.columns if c not in {"outcome", "sale", "outcome_binary"}]
    merged = ann_sub.merge(latent[latent_cols], on=["conversation_id", "turn"], how="inner")

    if "outcome" not in merged.columns or merged["outcome"].isna().all():
        if dataset_key == "heddaya_nego":
            merged["outcome"] = "sale"
        else:
            merged["outcome"] = np.nan
    merged["outcome"] = merged["outcome"].map(normalize_outcome)
    if dataset_key == "heddaya_nego":
        merged["outcome"] = "sale"
    merged["sale"] = (merged["outcome"] == "sale").astype(int)

    z_cols = find_z_cols(merged)
    if len(z_cols) == 0:
        raise ValueError(f"No z_* columns found in merged latent data for {dataset_key}")

    return merged, z_cols, out_dir

# -----------------------------
# Task 3B
# -----------------------------

def task_3b(merged, z_cols, min_turns=10):
    results = []
    act_cols = {
        "new_offer": "act_new_offer",
        "repeat_offer": "act_repeat",
        "push": "act_push",
        "push_constraint": None,
        "push_disparagement": None,
        "push_neutral": None,
        "comparison": "act_comparison",
        "comparison_price": None,
        "comparison_quality": None,
        "comparison_mixed": None,
        "allowance": "act_allowance",
        "end": "act_end",
    }

    z_lookup = merged.set_index(["conversation_id", "turn"])[z_cols]

    for act_name, col in act_cols.items():
        if col is not None and col in merged.columns:
            act_turns = merged[merged[col] == 1].copy()
        elif act_name.startswith("push_") and "push_subtype" in merged.columns:
            act_turns = merged[merged["push_subtype"] == act_name].copy()
        elif act_name.startswith("comparison_") and "comp_subtype" in merged.columns:
            act_turns = merged[merged["comp_subtype"] == act_name].copy()
        else:
            continue

        if len(act_turns) < min_turns:
            continue

        shifts, z_t_vals, z_t1_vals = [], [], []
        for _, row in act_turns.iterrows():
            key_t = (row["conversation_id"], int(row["turn"]))
            key_t1 = (row["conversation_id"], int(row["turn"]) + 1)
            if key_t in z_lookup.index and key_t1 in z_lookup.index:
                z_t = z_lookup.loc[key_t].values.astype(float)
                z_t1 = z_lookup.loc[key_t1].values.astype(float)
                shifts.append(z_t1 - z_t)
                z_t_vals.append(z_t)
                z_t1_vals.append(z_t1)

        if not shifts:
            continue

        shifts = np.asarray(shifts)
        z_t_arr = np.asarray(z_t_vals)
        z_t1_arr = np.asarray(z_t1_vals)
        mean_shift = shifts.mean(axis=0)
        dominant_dim = int(np.argmax(np.abs(mean_shift)) + 1)

        for di, zc in enumerate(z_cols):
            results.append({
                "act": act_name,
                "n_turns": len(shifts),
                "z_dim": zc,
                "mean_z_t": round(float(z_t_arr[:, di].mean()), 5),
                "mean_z_t1": round(float(z_t1_arr[:, di].mean()), 5),
                "mean_shift": round(float(mean_shift[di]), 5),
                "abs_shift": round(float(abs(mean_shift[di])), 5),
                "dominant_dim": f"z_{dominant_dim}",
            })

    return pd.DataFrame(results)

# -----------------------------
# Task 3C feature extraction
# -----------------------------

def get_prices_generic(text):
    prices = set()
    text = str(text)
    for m in re.finditer(r"\$\s*(\d[\d,\s\.]*)(?:\s*(k|K|thousand))?", text):
        raw = re.sub(r"[\s,]", "", m.group(1))
        try:
            val = float(raw)
            v = int(round(val if (m.group(2) or val < 1000) else val / 1000))
            if 5 <= v <= 300:
                prices.add(v)
        except Exception:
            pass
    for m in re.finditer(r"\b(\d{3})\b", text):
        try:
            v = int(m.group(1))
            if 150 <= v <= 300 and str(v) not in {"213", "233", "239"}:
                prices.add(v)
        except Exception:
            pass
    for m in re.finditer(r"\b(\d{5,6})\b", text):
        try:
            v = int(round(int(m.group(1)) / 1000))
            if 5 <= v <= 300:
                prices.add(v)
        except Exception:
            pass
    return sorted(prices)


def extract_price_features(merged):
    records = []
    for cid, grp in merged.groupby("conversation_id"):
        grp = grp.sort_values("turn")
        outcome = int(grp["sale"].iloc[0])
        buyer_prices, seller_prices = [], []

        for _, row in grp.iterrows():
            is_offer = int(row.get("act_new_offer", 0)) == 1 or int(row.get("act_allowance", 0)) == 1
            if is_offer:
                ps = get_prices_generic(row.get("text", ""))
                if ps:
                    if row.get("role") == "buyer":
                        buyer_prices.extend(ps)
                    else:
                        seller_prices.extend(ps)

        all_prices = buyer_prices + seller_prices
        default_mid = float(np.median(all_prices)) if all_prices else 0.0
        last_buyer = float(buyer_prices[-1]) if buyer_prices else default_mid
        last_seller = float(seller_prices[-1]) if seller_prices else default_mid
        first_buyer = float(buyer_prices[0]) if buyer_prices else last_buyer
        first_seller = float(seller_prices[0]) if seller_prices else last_seller

        records.append({
            "conversation_id": cid,
            "sale": outcome,
            "last_buyer": last_buyer,
            "last_seller": last_seller,
            "price_gap": abs(last_seller - last_buyer),
            "initial_gap": abs(first_seller - first_buyer),
            "buyer_movement": abs(last_buyer - first_buyer) if len(buyer_prices) > 1 else 0.0,
            "seller_movement": abs(last_seller - first_seller) if len(seller_prices) > 1 else 0.0,
            "n_offers": float(grp.get("act_new_offer", pd.Series(dtype=float)).sum()),
            "n_allowances": float(grp.get("act_allowance", pd.Series(dtype=float)).sum()),
            "midpoint": (last_buyer + last_seller) / 2 if (last_buyer or last_seller) else 0.0,
        })
    return pd.DataFrame(records)


def extract_ssm_features(merged, z_cols):
    records = []
    for cid, grp in merged.groupby("conversation_id"):
        grp = grp.sort_values("turn")
        Z = grp[z_cols].values.astype(float)
        T = len(Z)
        outcome = int(grp["sale"].iloc[0])
        if T == 0:
            continue
        z_mean = Z.mean(axis=0)
        z_std = Z.std(axis=0) + 1e-8
        z_last = Z[max(0, 2 * T // 3):].mean(axis=0)
        z_slope = np.array([np.polyfit(np.arange(T), Z[:, d], 1)[0] if T > 1 else 0.0 for d in range(len(z_cols))])

        buyer_z = grp[grp.get("role") == "buyer"][z_cols].values.astype(float) if "role" in grp.columns else np.empty((0, len(z_cols)))
        seller_z = grp[grp.get("role") == "seller"][z_cols].values.astype(float) if "role" in grp.columns else np.empty((0, len(z_cols)))
        min_len = min(len(buyer_z), len(seller_z))
        divergence = np.abs(buyer_z[:min_len] - seller_z[:min_len]).mean(axis=0) if min_len > 0 else np.zeros(len(z_cols))

        rec = {"conversation_id": cid, "sale": outcome}
        for i, zc in enumerate(z_cols):
            rec[f"{zc}_mean"] = z_mean[i]
            rec[f"{zc}_std"] = z_std[i]
            rec[f"{zc}_last"] = z_last[i]
            rec[f"{zc}_slope"] = z_slope[i]
            rec[f"{zc}_diverge"] = divergence[i]
        records.append(rec)
    return pd.DataFrame(records)


def extract_tactic_features(merged):
    records = []
    act_cols = ["act_new_offer", "act_repeat", "act_push", "act_comparison", "act_allowance", "act_end"]
    for cid, grp in merged.groupby("conversation_id"):
        T = max(len(grp), 1)
        rec = {"conversation_id": cid, "sale": int(grp["sale"].iloc[0])}
        for col in act_cols:
            rec[col.replace("act_", "")] = grp[col].sum() / T if col in grp.columns else 0.0
        for sub_t in ["push_constraint", "push_disparagement", "push_neutral"]:
            rec[sub_t] = (grp["push_subtype"] == sub_t).sum() / T if "push_subtype" in grp.columns else 0.0
        for sub_t in ["comparison_price", "comparison_quality", "comparison_mixed"]:
            rec[sub_t] = (grp["comp_subtype"] == sub_t).sum() / T if "comp_subtype" in grp.columns else 0.0

        buyer = grp[grp["role"] == "buyer"] if "role" in grp.columns else pd.DataFrame()
        seller = grp[grp["role"] == "seller"] if "role" in grp.columns else pd.DataFrame()
        for prefix, part in [("buyer", buyer), ("seller", seller)]:
            denom = max(len(part), 1)
            rec[f"{prefix}_push_rate"] = part["act_push"].sum() / denom if "act_push" in part.columns else 0.0
            rec[f"{prefix}_comparison_rate"] = part["act_comparison"].sum() / denom if "act_comparison" in part.columns else 0.0
            rec[f"{prefix}_allowance_rate"] = part["act_allowance"].sum() / denom if "act_allowance" in part.columns else 0.0

        rec["push_then_comp"] = (((grp.get("act_push", 0) == 1) & (grp.get("act_comparison", 0) == 1)).sum() / T)
        records.append(rec)
    return pd.DataFrame(records)


def safe_horse_race(price_df, ssm_df, tactic_df, n_splits=5):
    base = price_df[["conversation_id", "sale"]].copy()
    df = base.merge(price_df.drop("sale", axis=1), on="conversation_id")
    df = df.merge(ssm_df.drop("sale", axis=1), on="conversation_id")
    df = df.merge(tactic_df.drop("sale", axis=1), on="conversation_id")

    y = df["sale"].values.astype(int)
    price_feats = [c for c in price_df.columns if c not in ["conversation_id", "sale"]]
    ssm_feats = [c for c in ssm_df.columns if c not in ["conversation_id", "sale"]]
    tactic_feats = [c for c in tactic_df.columns if c not in ["conversation_id", "sale"]]
    models = {
        "M1_price": price_feats,
        "M2_ssm": ssm_feats,
        "M3_price_ssm": price_feats + ssm_feats,
        "M4_full": price_feats + ssm_feats + tactic_feats,
    }

    if len(np.unique(y)) < 2:
        empty = pd.DataFrame([
            {"model": name, "mean_cv_auc": np.nan, "std_cv_auc": np.nan, "overall_auc": np.nan,
             "n_features": len(feats), "status": "skipped_single_outcome_class"}
            for name, feats in models.items()
        ]).set_index("model")
        return empty, df, models

    class_counts = pd.Series(y).value_counts()
    max_splits = int(class_counts.min())
    n_splits = min(n_splits, max_splits)
    if n_splits < 2:
        empty = pd.DataFrame([
            {"model": name, "mean_cv_auc": np.nan, "std_cv_auc": np.nan, "overall_auc": np.nan,
             "n_features": len(feats), "status": "skipped_too_few_minority_cases"}
            for name, feats in models.items()
        ]).set_index("model")
        return empty, df, models

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1)
    results = []
    for model_name, feats in models.items():
        F = df[feats].fillna(0).replace([np.inf, -np.inf], 0).values
        F_sc = StandardScaler().fit_transform(F)
        aucs = []
        preds = np.zeros(len(y))
        for tr, te in cv.split(F_sc, y):
            clf.fit(F_sc[tr], y[tr])
            prob = clf.predict_proba(F_sc[te])[:, 1]
            preds[te] = prob
            aucs.append(roc_auc_score(y[te], prob))
        results.append({
            "model": model_name,
            "mean_cv_auc": round(float(np.mean(aucs)), 4),
            "std_cv_auc": round(float(np.std(aucs)), 4),
            "overall_auc": round(float(roc_auc_score(y, preds)), 4),
            "n_features": len(feats),
            "status": "ok",
        })
    return pd.DataFrame(results).set_index("model"), df, models


def run_dataset(dataset_key):
    diag = []
    def log(s=""):
        print(s)
        diag.append(str(s))

    log("=" * 72)
    log(f"TASKS 3B + 3C — {dataset_key}")
    log("=" * 72)

    out_dir = f"{OUTPUT_ROOT}/{dataset_key}"
    os.makedirs(out_dir, exist_ok=True)

    try:
        merged, z_cols, _ = load_merged(dataset_key)
    except Exception as e:
        log(f"SKIPPED: {e}")
        with open(f"{out_dir}/task3bc_diagnostics.txt", "w") as f:
            f.write("\n".join(diag))
        return None

    log(f"Merged dataset : {merged.shape}")
    log(f"Conversations  : {merged['conversation_id'].nunique():,}")
    log(f"z dimensions   : {len(z_cols)} ({', '.join(z_cols[:5])}{'...' if len(z_cols) > 5 else ''})")
    log(f"Sale rate      : {merged.groupby('conversation_id')['sale'].first().mean():.1%}")
    log(f"Outcome classes: {merged.groupby('conversation_id')['sale'].first().value_counts().to_dict()}")

    log("\n[Task 3B] Tactic → latent state shifts")
    shift_df = task_3b(merged, z_cols)
    shift_df.to_csv(f"{out_dir}/task3b_tactic_shifts.csv", index=False)
    log(f"  Saved task3b_tactic_shifts.csv: {shift_df.shape}")

    if len(shift_df):
        summary = shift_df.loc[shift_df.groupby("act")["abs_shift"].idxmax()].reset_index(drop=True)
        log("\n  Dominant shift per act:")
        for _, row in summary.sort_values("abs_shift", ascending=False).head(12).iterrows():
            arrow = "↑" if row["mean_shift"] > 0 else "↓"
            log(f"    {row['act']:<24} {row['z_dim']:<5} shift={row['mean_shift']:+.5f}{arrow} n={int(row['n_turns'])}")

    log("\n[Task 3C] Feature extraction and horse-race regression")
    price_df = extract_price_features(merged)
    ssm_df = extract_ssm_features(merged, z_cols)
    tactic_df = extract_tactic_features(merged)
    horse_results, feat_df, models = safe_horse_race(price_df, ssm_df, tactic_df)

    feat_df.to_csv(f"{out_dir}/task3c_features.csv", index=False)
    horse_results.to_csv(f"{out_dir}/task3c_horse_race.csv")

    log(f"  Price features : {len([c for c in price_df.columns if c not in ['conversation_id','sale']])}")
    log(f"  SSM features   : {len([c for c in ssm_df.columns if c not in ['conversation_id','sale']])}")
    log(f"  Tactic features: {len([c for c in tactic_df.columns if c not in ['conversation_id','sale']])}")
    log("\n  Horse-race results:")
    for model_name, row in horse_results.iterrows():
        log(f"    {model_name:<14} AUC={row['mean_cv_auc']} overall={row['overall_auc']} features={int(row['n_features'])} status={row['status']}")

    if dataset_key == "heddaya_nego" and horse_results["status"].iloc[0] == "skipped_single_outcome_class":
        log("\n  Note: Heddaya outcomes were assumed to be all sale. AUC is skipped because ROC-AUC requires both sale and no-sale classes.")

    with open(f"{out_dir}/task3bc_diagnostics.txt", "w") as f:
        f.write("\n".join(diag))

    log("=" * 72)
    log(f"TASKS 3B + 3C COMPLETE — {dataset_key}")
    log("=" * 72)
    return {"shift_df": shift_df, "horse_results": horse_results, "features": feat_df}


def main():
    results = {}
    for dataset_key in DATASETS:
        results[dataset_key] = run_dataset(dataset_key)
        print()
    return results

if __name__ == "__main__":
    results = main()
